# Article 1 — SAM 3 Transformers video runner

This notebook follows the official Hugging Face `facebook/sam3` video API. It uses `Sam3VideoModel` and `Sam3VideoProcessor`, not the separate checkpoint-only SAM 3.1 Object Multiplex release.

It runs on Colab or Kaggle, keeps decoded video frames on CPU, uses FP16 on a T4, and applies Hugging Face's documented 560-pixel lower-memory configuration. The default smoke test downloads the same public bedroom MP4 used in the official documentation and prompts for `person`.

For a controlled comparison with Grounded SAM 2, set `USE_OFFICIAL_SAMPLE = False` and use the same source MP4 and semantic prompt in both notebooks.


In [ ]:
import platform
import subprocess
import sys

print(f"Python: {platform.python_version()}")
gpu_report = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"GPU: {gpu_report}")


In [ ]:
from base64 import b64encode
from pathlib import Path

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
GVI = ROOT / "grounded-visual-intelligence"


def get_secret(name: str) -> str | None:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)


github_token = get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Add a read-only GITHUB_TOKEN while the article repository is private")
basic = b64encode(f"x-access-token:{github_token}".encode()).decode()

if not GVI.exists():
    subprocess.run(
        [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {basic}",
            "clone",
            "--branch",
            "article/01-grounded-video-memory",
            "--depth",
            "1",
            "https://github.com/vlada22/grounded-visual-intelligence.git",
            str(GVI),
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=5.11,<6",
        "accelerate>=1.0",
        "av>=14",
        "-e",
        str(GVI),
    ],
    check=True,
)
src_dir = str(GVI / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from gvi.adapters.sam3 import Sam3Adapter

print(f"Dependencies installed; repository package import verified from {src_dir}")


In [ ]:
from huggingface_hub import HfApi, login

MODEL_ID = "facebook/sam3"
hf_token = get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add the approved HF_TOKEN to Colab or Kaggle Secrets")
login(token=hf_token, add_to_git_credential=False)
try:
    model_info = HfApi().model_info(MODEL_ID, token=hf_token)
except Exception as error:
    raise RuntimeError(f"HF_TOKEN cannot access {MODEL_ID}") from error
print(f"Hugging Face access ready: {model_info.id}")


In [ ]:
import time

from huggingface_hub import hf_hub_download

from gvi.run_artifacts import sha256_file

USE_OFFICIAL_SAMPLE = True
PROMPT = "person"
SOURCE_URI = "hf://datasets/hf-internal-testing/sam2-fixtures/bedroom.mp4"

if USE_OFFICIAL_SAMPLE:
    source_download_started = time.perf_counter()
    source_video_path = Path(
        hf_hub_download(
            repo_id="hf-internal-testing/sam2-fixtures",
            repo_type="dataset",
            filename="bedroom.mp4",
        )
    )
    source_download_s = time.perf_counter() - source_download_started
else:
    if Path("/kaggle/input").exists():
        candidates = sorted(Path("/kaggle/input").rglob("*.mp4"))
        if not candidates:
            raise RuntimeError("Add an MP4 as a Kaggle dataset or switch to the official sample")
        source_video_path = candidates[0]
    else:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one MP4")
        source_video_path = ROOT / next(iter(uploaded))
    PROMPT = "red toy car"
    SOURCE_URI = f"uploaded://{source_video_path.name}"
    source_download_s = 0.0

video_path = ROOT / "article-01-comparison.mp4"
preprocess_started = time.perf_counter()
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        str(source_video_path),
        "-t",
        "3",
        "-vf",
        "fps=4,scale='min(560,iw)':-2:flags=lanczos",
        "-an",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        str(video_path),
    ],
    check=True,
)
preprocess_s = time.perf_counter() - preprocess_started
source_sha256 = sha256_file(source_video_path)
prepared_sha256 = sha256_file(video_path)
print(
    {
        "source": source_video_path.name,
        "prepared": video_path.name,
        "prompt": PROMPT,
        "source_sha256": source_sha256,
        "prepared_sha256": prepared_sha256,
    }
)


In [ ]:
import json
from fractions import Fraction

probe = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height,avg_frame_rate,nb_frames,duration",
        "-of",
        "json",
        str(video_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
stream = json.loads(probe.stdout)["streams"][0]
fps = float(Fraction(stream["avg_frame_rate"]))
duration = float(stream.get("duration") or 0)
frame_count = int(stream.get("nb_frames") or round(duration * fps))
video_info = {
    "width": int(stream["width"]),
    "height": int(stream["height"]),
    "fps": fps,
    "frame_count": frame_count,
}
if fps <= 0 or frame_count <= 0:
    raise RuntimeError("Prepared video metadata is invalid")
video_info


In [ ]:
import time

import torch
import transformers
from huggingface_hub import snapshot_download
from transformers import Sam3VideoConfig, Sam3VideoModel, Sam3VideoProcessor
from transformers.video_utils import load_video

print(f"Transformers: {transformers.__version__}")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model_download_started = time.perf_counter()
model_path = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        token=hf_token,
        allow_patterns=[
            "*.json",
            "*.txt",
            "*.safetensors",
            "LICENSE",
            "README.md",
        ],
    )
)
model_download_s = time.perf_counter() - model_download_started
model_revision = model_path.name

config = Sam3VideoConfig.from_pretrained(model_path)
config.image_size = 560

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model_load_started = time.perf_counter()
model = Sam3VideoModel.from_pretrained(
    model_path,
    config=config,
    device_map="auto",
    dtype=amp_dtype,
)
processor = Sam3VideoProcessor.from_pretrained(
    model_path,
    size={"height": 560, "width": 560},
)
torch.cuda.synchronize()
model_load_s = time.perf_counter() - model_load_started
inference_device = next(model.parameters()).device

video_decode_started = time.perf_counter()
video_frames, _ = load_video(str(video_path))
video_decode_s = time.perf_counter() - video_decode_started

session_started = time.perf_counter()
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=inference_device,
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=amp_dtype,
)
session_init_s = time.perf_counter() - session_started

prompt_started = time.perf_counter()
inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text=PROMPT,
)
torch.cuda.synchronize()
prompt_s = time.perf_counter() - prompt_started

outputs_per_frame = []
torch.cuda.synchronize()
inference_started = time.perf_counter()
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session,
    max_frame_num_to_track=max(0, len(video_frames) - 1),
):
    processed = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame.append((int(model_outputs.frame_idx), processed))
torch.cuda.synchronize()
inference_s = time.perf_counter() - inference_started

{
    "model_revision": model_revision,
    "device_map": getattr(model, "hf_device_map", None),
    "model_download_s": model_download_s,
    "model_load_s": model_load_s,
    "video_decode_s": video_decode_s,
    "session_init_s": session_init_s,
    "prompt_s": prompt_s,
    "inference_s": inference_s,
    "frames": len(outputs_per_frame),
}


In [ ]:
import numpy as np

artifact_export_started = time.perf_counter()

from gvi.adapters.sam3 import (
    NormalizedBoxXYWH,
    Sam3Adapter,
    Sam3FrameRecord,
    Sam3RecordedOutput,
)
from gvi.inference.mask_rle import encode_binary_mask

SCENE_ID = "article-01-hero-sam3-transformers"
OUTPUT_DIR = ROOT / "gvi-artifacts" / SCENE_ID
MASK_DIR = OUTPUT_DIR / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)


def cpu_array(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    return np.asarray(value)


frame_records = []
for frame_index, output in outputs_per_frame:
    object_ids = cpu_array(output["object_ids"]).reshape(-1)
    scores = cpu_array(output["scores"]).reshape(-1)
    boxes = cpu_array(output["boxes"]).reshape(-1, 4)
    masks = cpu_array(output["masks"])
    if masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]
    masks = masks.astype(bool)

    kept_ids = []
    kept_scores = []
    kept_boxes = []
    kept_mask_uris = []
    for object_id, score, box, mask in zip(
        object_ids, scores, boxes, masks, strict=True
    ):
        x_min, y_min, x_max, y_max = np.asarray(box, dtype=float)
        x_min = float(np.clip(x_min, 0, video_info["width"]))
        x_max = float(np.clip(x_max, 0, video_info["width"]))
        y_min = float(np.clip(y_min, 0, video_info["height"]))
        y_max = float(np.clip(y_max, 0, video_info["height"]))
        if x_max <= x_min or y_max <= y_min or not mask.any():
            continue

        object_id = int(object_id)
        relative_mask = Path("masks") / (
            f"frame-{frame_index:05d}-object-{object_id}.rle.json"
        )
        (OUTPUT_DIR / relative_mask).write_text(
            encode_binary_mask(mask).model_dump_json(indent=2),
            encoding="utf-8",
        )
        kept_ids.append(object_id)
        kept_scores.append(float(score))
        kept_boxes.append(
            NormalizedBoxXYWH(
                x=x_min / video_info["width"],
                y=y_min / video_info["height"],
                width=(x_max - x_min) / video_info["width"],
                height=(y_max - y_min) / video_info["height"],
            )
        )
        kept_mask_uris.append(relative_mask.as_posix())

    frame_records.append(
        Sam3FrameRecord(
            frame_index=frame_index,
            out_obj_ids=tuple(kept_ids),
            out_probs=tuple(kept_scores),
            out_boxes_xywh=tuple(kept_boxes),
            out_mask_uris=tuple(kept_mask_uris),
        )
    )

recorded = Sam3RecordedOutput(
    scene_id=SCENE_ID,
    source_uri=SOURCE_URI,
    prompt=PROMPT,
    model_id=MODEL_ID,
    model_version=f"transformers-{transformers.__version__}",
    width=video_info["width"],
    height=video_info["height"],
    fps=video_info["fps"],
    frame_count=video_info["frame_count"],
    frames=tuple(frame_records),
)
(OUTPUT_DIR / "sam3-output.json").write_text(
    recorded.model_dump_json(indent=2), encoding="utf-8"
)
scene = Sam3Adapter().convert(recorded)
(OUTPUT_DIR / "scene.json").write_text(
    scene.model_dump_json(indent=2), encoding="utf-8"
)
artifact_export_s = time.perf_counter() - artifact_export_started
metrics = {
    "pipeline": "sam3-transformers",
    "runtime": "kaggle" if Path("/kaggle/working").exists() else "colab",
    "runtime_metadata": None,
    "model_revision": model_revision,
    "source_uri": SOURCE_URI,
    "source_sha256": source_sha256,
    "prepared_sha256": prepared_sha256,
    "source_download_s": source_download_s,
    "preprocess_s": preprocess_s,
    "model_download_s": model_download_s,
    "video_decode_s": video_decode_s,
    "session_init_s": session_init_s,
    "prompt_s": prompt_s,
    "artifact_export_s": artifact_export_s,
    "model_id": MODEL_ID,
    "prompt": PROMPT,
    "dtype": str(amp_dtype),
    "comparison_preset": {"seconds": 3, "fps": 4, "max_width": 560},
    "model_load_s": model_load_s,
    "inference_s": inference_s,
    "total_model_s": model_load_s + session_init_s + prompt_s + inference_s,
    "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "tracked_frames": len(frame_records),
    "source_frames": video_info["frame_count"],
}
metrics


In [ ]:
import shutil

from gvi.run_artifacts import build_analysis_artifacts, runtime_metadata, write_manifest

analysis_started = time.perf_counter()
analysis_summary = build_analysis_artifacts(
    output_dir=OUTPUT_DIR,
    video_path=video_path,
    scene_path=OUTPUT_DIR / "scene.json",
)
analysis_artifact_s = time.perf_counter() - analysis_started
metrics["analysis_artifact_s"] = analysis_artifact_s
metrics["runtime_metadata"] = runtime_metadata(torch)
metrics["peak_gpu_memory_allocated_gib"] = torch.cuda.max_memory_allocated() / 1024**3
metrics["peak_gpu_memory_reserved_gib"] = torch.cuda.max_memory_reserved() / 1024**3
metrics["analysis_summary"] = analysis_summary
(OUTPUT_DIR / "run-metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
manifest = write_manifest(OUTPUT_DIR)

archive_started = time.perf_counter()
archive = shutil.make_archive(str(ROOT / SCENE_ID), "zip", OUTPUT_DIR)
archive_s = time.perf_counter() - archive_started
print(
    {
        "artifact_bundle": archive,
        "archive_s": archive_s,
        "files": len(manifest["files"]),
        "archive_sha256": sha256_file(Path(archive)),
    }
)
if not Path("/kaggle/working").exists():
    from google.colab import files

    files.download(archive)


## Expected result

The run produces `article-01-hero-sam3-transformers.zip`. It contains the
prepared comparison clip, annotated preview video, contact sheet, per-frame track
measurements, analysis summary, integrity manifest, raw normalized recording,
model-independent `scene.json`, RLE masks, and detailed phase/runtime metrics.
On Kaggle, download it from the notebook's Output/Files panel. On Colab, the
download starts automatically.
